# PySpark job: resample and join the largest table

This notebook runs **the same stages as `src/08_spark_job.py`**, one cell at a time, so the output of each stage (and its row count) is saved here. The brief asks for *row counts before and after* every stage: see the last cell.

Re-run everything with `make spark` (it executes this notebook and saves the outputs).

In [ ]:
# Load the script as a module, so the notebook and the script can never drift apart.
import sys, importlib.util
sys.path.insert(0, '../src')
spec = importlib.util.spec_from_file_location('job', '../src/08_spark_job.py')
job = importlib.util.module_from_spec(spec)
spec.loader.exec_module(job)

job.reset_cleaning_log_for_stage(job.STAGE)  # avoid duplicate log rows when both script and notebook run
spark = job.start_spark()
print('Spark', spark.version)

## Stage 1: load
The full grid from `07_gaps_outliers.py`: every station × parameter × hour, including empty hours.

In [ ]:
readings = job.stage_load(spark)
readings.printSchema()
readings.groupBy('qc_flag').count().orderBy('count', ascending=False).show()

## Stage 2: clean
Drop hours with no value and every sensor-failure flag. Real extreme events (`real_event_*`) stay in.

In [ ]:
clean = job.stage_clean(readings)

## Stage 3: resample to hourly
`date_trunc('hour')` + `avg()`. Our data is already hourly, so the count should not change. That is the check.

In [ ]:
hourly = job.stage_resample(clean)
hourly.show(5)

## Stage 4: join with weather and the calendar
Left joins, so no air-quality hour is ever silently dropped.

In [ ]:
joined = job.stage_join(spark, hourly)
joined.select('station_id', 'ts_hour', 'value', 'precip_mm', 'is_rain_hour', 'weekday', 'is_holiday', 'season').show(5)

## Stage 5: aggregate (hour × weekday, month × hour, rain × hour)

In [ ]:
results = job.stage_aggregate(joined)
results['spark_pm25_hour_weekday.csv'].filter('hour >= 17').show(14)

## Row counts after every stage

In [ ]:
import pandas as pd
row_counts = pd.DataFrame(job.ROW_COUNTS)
row_counts.to_csv(job.TABLES / 'spark_row_counts.csv', index=False)
row_counts

In [ ]:
spark.stop()